[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flexfengfeng/dsai-m3-ml-genai/blob/main/L10-transformers-genai/notebooks/04_rag_pipeline.ipynb)

**Where to run this notebook**

- **Locally (VS Code + Jupyter)**: just open the notebook and pick the `dsai-m3` kernel. The repo's `.env` file handles thread caps automatically.
- **Colab (recommended if you don't have a GPU)**: click the badge above, then **Runtime → Change runtime type → T4 GPU**, then run the setup cell below. It clones the repo, installs missing deps, and `cd`s into the right working directory.


In [1]:
# === Colab-compat setup (no-op when running locally) ===
# This whole `if IN_COLAB:` block does NOTHING on your laptop — it only fires
# on Google Colab, where it clones the course repo, installs the missing
# libraries, and moves into the right folder so the data/ paths resolve.
import os, sys

IN_COLAB = "google.colab" in sys.modules  # True only inside a Colab runtime

if IN_COLAB:
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI.git"
    REPO_DIR = "/content/6m-data-3.10-Transformers-Attention-GenAI"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # Colab has torch + torchvision pre-installed. Install the rest.
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

# Threading caps — picked up by .env locally, set explicitly here as a fallback.
# Harmless if already set. (Loop form prevents Jupyter from auto-displaying the return value.)
for _key, _val in [("OMP_NUM_THREADS", "1"), ("MKL_NUM_THREADS", "1"), ("TOKENIZERS_PARALLELISM", "false")]:
    os.environ.setdefault(_key, _val)

# L10 · NB 04 — RAG: Retrieval-Augmented Generation

> *We have embeddings (L09). We have an LLM (NB 03). Let's stitch them together into the architecture behind every modern "chat with your data" product — including Marcus's shopping assistant.*

In this notebook we will:

1. Show why an LLM **alone** can't answer questions about NorthStar's products (it doesn't know them — it hallucinates)
2. Build a retrieval step using `all-MiniLM-L6-v2` (from L09)
3. Stuff retrieved products into a prompt and send it to SmolLM2-360M
4. Watch the LLM produce a grounded answer with real product names
5. Build a clean `RAGSystem` class — what you'd actually ship

This is the closing pattern of the entire module.

---

### Where you are in L10

Marcus asked for a shopping assistant. Sarah can't build it in one leap — she works through four notebooks, each answering the next question in the chain:

| Step | Notebook | Sarah's question |
|---|---|---|
| 1 | `01_monday_morning` | What can pretrained transformers already do, out of the box? |
| 2 | `02_attention_intuition` | *Why* do they work? What is this "attention" everyone talks about? |
| 3 | `03_using_an_llm` | How do I drive a generative LLM myself — tokens, sampling, chat? |
| 4 | `04_rag_pipeline` 👈 **you are here** | How do I ground it in NorthStar's catalogue? → the shopping assistant |

Right now you're on **step 4 of 4**. Each notebook stands on the one before it — run them in order.


## 1 · Setup


> **Why this cell first — and why it looks like a pile of switches.**
> Before Sarah touches a single model she runs this setup cell. It isn't glamorous, but every line earns its place:
>
> - `KMP_DUPLICATE_LIB_OK='TRUE'` — on macOS, PyTorch and the other scientific libraries each ship their own copy of the OpenMP maths runtime. When two copies load at once the kernel **crashes the instant you `import torch`**. This line tells them to coexist instead of fighting. (It's the same fix the repo's `.env` file applies — we repeat it here so the notebook also works in Colab and on a fresh machine.)
> - `torch.set_num_threads(1)` — pins the model to a single CPU thread. On a laptop this is *more* stable and predictable than letting it grab every core.
> - `HF_HUB_DISABLE_TELEMETRY` / `TRANSFORMERS_VERBOSITY='error'` / `warnings.filterwarnings('ignore')` — silence the download chatter and version warnings so the output you see is the output that *matters*.
>
> You'll see this same opening cell in every model notebook. Run it once at the top — it's the seatbelt before the drive.


In [2]:
# --- Stability + quiet-output environment switches (set BEFORE importing torch) ---
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'  # don't phone home when downloading models
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'  # hide info/warning chatter from transformers

import warnings
warnings.filterwarnings('ignore')  # silence version/deprecation warnings so real output stands out

# --- Core libraries (grouped): numerics, data tables, deep-learning, the two model families ---
import time                                                    # to time how long generation takes
import numpy as np                                             # array math (argsort, indexing)
import pandas as pd                                            # load + slice the catalogue CSV
import torch                                                   # the tensor/deep-learning backend
from sentence_transformers import SentenceTransformer          # the EMBEDDING (retrieval) model
from sklearn.metrics.pairwise import cosine_similarity         # measures how similar two embeddings are
from transformers import AutoTokenizer, AutoModelForCausalLM   # the GENERATIVE LLM + its tokenizer

torch.set_num_threads(1)  # pin to 1 CPU thread — more stable/predictable on a laptop
torch.manual_seed(0)      # fix the random seed so results are reproducible run to run

# RAG needs TWO models: one to RETRIEVE (find relevant products), one to GENERATE (write the answer).
print('Loading retrieval model...')
retriever = SentenceTransformer('all-MiniLM-L6-v2')  # turns text into 384-dim vectors (from L09)

print('Loading LLM (SmolLM2-360M-Instruct)...')
LLM_NAME = 'HuggingFaceTB/SmolLM2-360M-Instruct'      # a small instruction-tuned chat model
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)   # converts text <-> token IDs
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME)  # the model that predicts the next token
print('Both models ready.')

Loading retrieval model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading LLM (SmolLM2-360M-Instruct)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both models ready.


## 2 · Load NorthStar's catalogue

In [3]:
# Load NorthStar's product catalogue from CSV into a pandas DataFrame (a table).
# Each row is one product; columns include name, category, description, price_gbp.
df = pd.read_csv('data/northstar_catalogue.csv')
# .nunique() counts the distinct categories; len(df) is the number of products (rows).
print(f"Catalogue: {len(df)} products across {df['category'].nunique()} categories")
df.head(5)  # peek at the first 5 rows to see the column layout

Catalogue: 76 products across 10 categories


,product_id,name,category,description,price_gbp
0,P0001,Lila Floral Sundress,dress,Lightweight floral frock perfect for warm summ...,49
1,P0002,Marina Wrap Dress,dress,Elegant wrap-style dress in deep navy. Suitabl...,89
2,P0003,Cassia Maxi Gown,dress,Flowing full-length gown with adjustable strap...,65
3,P0004,Holly Knit Jumper Dress,dress,Cosy ribbed knit dress in oatmeal. Perfect win...,72
4,P0005,Ivy Slip Dress,dress,Minimalist satin slip in champagne. Layer over...,55


## 3 · Why an LLM alone fails: the hallucination problem

Let's ask the LLM about NorthStar products WITHOUT giving it the catalogue. It will produce something plausible — but completely invented.

**<span style="color:green">[Opus 4.8]</span> Why LLMs "hallucinate," in plain English:**

It's tempting to think the model is *lying* or *guessing badly*. It isn't — it's doing the **only thing it was ever trained to do: predict plausible next words.** An LLM has no built-in database of facts and no way to check anything against reality. It learned the *shape* of language — what a confident product recommendation tends to sound like — from the internet.

So when you ask it about **NorthStar's catalogue**, which it has never seen, there's no fact to retrieve. But the model can't answer "I don't know" unless that's the most likely continuation — and after a helpful "Sure, here are some options:" the most likely continuation is… **a list of plausible-sounding products.** So it confidently invents names, prices, and descriptions that *fit the pattern* of a real answer. This is a **hallucination**: fluent, well-formed, and completely made up.

The two traps that make this dangerous:
- **It sounds exactly as confident when it's wrong as when it's right.** Fluency is not knowledge. There's no wobble in its voice to warn you.
- **It's not fixable by "asking nicely" or fine-tuning harder.** The problem isn't the model's attitude; it's that the *facts aren't in the room.* You can't recall information that was never provided.

Which points straight at the fix (next section): if the model can only work with what's in its prompt, then **put the real catalogue in the prompt.** Don't ask it to remember NorthStar's products — *show* them to it, every time. That's RAG.

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

In [4]:
def llm_chat(messages, max_new_tokens=200, do_sample=False, repetition_penalty=1.15):
    """Send a chat-formatted prompt to the LLM.
    repetition_penalty > 1 prevents the model from getting stuck in degenerate
    loops on long contexts — important for a 360M-parameter model handling
    structured catalogue data.
    """
    # Turn the list of role/content messages into the exact text format the model was trained on.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Encode that prompt text into token IDs (a tensor the model can consume).
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
    # Generate new tokens. do_sample=False => greedy/deterministic output.
    out = llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=do_sample,
                       pad_token_id=tokenizer.eos_token_id,
                       repetition_penalty=repetition_penalty)
    # Slice off the prompt tokens (input_ids.shape[1]) so we return ONLY the model's new reply.
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

# Ask the LLM about NorthStar products with no context
question = "I'm looking for a summer dress under £80. What does NorthStar have?"
# Note: there is NO catalogue here — just a system role + the user's question.
naive_messages = [
    {'role': 'system', 'content': 'You are a helpful retail shopping assistant for NorthStar.'},
    {'role': 'user', 'content': question},
]

print(f"Q: {question}\n")
print(f"A (LLM alone, no catalogue):")
t0 = time.time()
# The model has no real product data, so it will INVENT plausible-sounding products (hallucination).
print(llm_chat(naive_messages))
print(f"\n[generated in {time.time()-t0:.1f}s]")

Q: I'm looking for a summer dress under £80. What does NorthStar have?

A (LLM alone, no catalogue):
NorthStar has several stylish and affordable options that fit your criteria:

1. **Luxury Dress**: A high-end, designer dress can cost upwards of £250-$300 per season. However, you might find some unique pieces at lower prices in the northstar_shop or online stores like Zara, H&M, or ASOS. 

2. **Midi Dress**: This is a mid-length skirt with a full hemline. It's often more budget-friendly than a mini dress but still offers good style points. You could look into brands like Gucci, Prada, or Chanel to see if they offer any discounts on their midi dresses.

3. **Dress with Leather**: If you're interested in something classic and elegant, consider a leather dress. They usually start from around £60-£70. Look out for brands like Dolce & Gabbana, Louis Vuitton, or Herm

[generated in 25.9s]


**Read that carefully.** The model produced a confident, fluent-sounding response. Almost everything in it is invented — fabricated product names, made-up prices, hallucinated descriptions.

This is the LLM doing exactly what it was trained to do (produce plausible text) with no grounding. It has no idea what NorthStar actually sells.

You CANNOT ship this. A customer who clicks on a recommended product and finds it doesn't exist loses trust forever. We need to fix the hallucination.

## 4 · The fix: retrieve first, then generate

The recipe:
1. Embed the catalogue once (L09)
2. Embed the query
3. Find the top-K most relevant products
4. Include them in the LLM's prompt
5. Ask the LLM to answer ONLY using the retrieved products

**<span style="color:green">[Opus 4.8]</span> RAG in plain English — the open-book exam:**

Here's the whole idea in one image. Asking the LLM about NorthStar with no catalogue is a **closed-book exam**: the student is fluent and confident but is working from memory alone, so they make things up. **RAG turns it into an open-book exam** — right before answering, you hand the student the *exact pages* they need, and say "answer using only these."

The name spells out the three steps (**R**etrieval-**A**ugmented **G**eneration):

| step | plain English | what it uses |
|---|---|---|
| **Retrieve** | Find the handful of catalogue rows most relevant to the question | the **embedding** search from L09 — turn the question and every product into vectors, grab the closest matches |
| **Augment** | Paste those rows into the prompt, with an instruction: *"recommend only from these"* | just string-building — the message list from NB 03 |
| **Generate** | Let the LLM read the question **and** the pasted rows, and write a natural-language answer | the LLM from NB 03 |

Why split the work this way instead of just using a smarter model? Because the two halves are good at *different* things. **Retrieval brings the facts** — it's cheap, fast, and grounded in your real data, and it gives you an **audit trail** (you can always see which rows were used). **The LLM brings the language** — it reads the rows and phrases a fluent, helpful recommendation. Neither could do the job alone: retrieval can't write a sentence, and the LLM doesn't know your catalogue. Bolt them together and you get fluent answers that are also *true* — the pattern behind essentially every "chat with your documents" product.

Notice what RAG is **not**: it's not training or fine-tuning. The models never change. You're just **feeding the right context at question time** — which, from NB 03, you know is the only thing the model actually "knows."

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

In [5]:
# Step 1 + 2: embed the catalogue (once) and the query (per request)
print('Embedding catalogue...')
# Build one descriptive string per product (name + description + price) — richer text embeds better.
docs = (df['name'] + ' — ' + df['description'] + f' (£' + df['price_gbp'].astype(str) + ')').tolist()
# encode() turns each product string into a fixed-length numeric vector capturing its meaning.
catalogue_emb = retriever.encode(docs, show_progress_bar=False)
# Shape is (76, 384): 76 products (rows), each a 384-dimensional embedding (all-MiniLM-L6-v2's size).
print(f"Catalogue embeddings shape: {catalogue_emb.shape}")

Embedding catalogue...
Catalogue embeddings shape: (76, 384)


In [6]:
def retrieve(query, top_k=5):
    """Return the top-K most relevant products as a DataFrame."""
    # Embed the query into the SAME 384-dim space as the catalogue so they're comparable.
    q_emb = retriever.encode([query])
    # cosine_similarity scores the query against every product (0..1); higher = closer in meaning.
    sims = cosine_similarity(q_emb, catalogue_emb)[0]
    # argsort sorts ascending, so negate sims to get the highest scores first; take the top_k.
    top_idx = np.argsort(-sims)[:top_k]
    # Pull those rows from the DataFrame by position.
    out = df.iloc[top_idx].copy()
    out['similarity'] = sims[top_idx]  # attach the score so we can inspect relevance
    return out.reset_index(drop=True)

# Test the retrieval for our question
question = "I'm looking for a summer dress under £80. What does NorthStar have?"
retrieved = retrieve(question, top_k=5)  # the 5 catalogue products closest in meaning to the query
print(f"Top {len(retrieved)} products retrieved:")
print(retrieved[['product_id','name','category','price_gbp','similarity']].to_string(index=False))

Top 5 products retrieved:
product_id                    name category  price_gbp  similarity
     P0007    Sienna Bodycon Dress    dress         45    0.583023
     P0010    Marigold Shift Dress    dress         52    0.561665
     P0004 Holly Knit Jumper Dress    dress         72    0.542383
     P0031     Verge Bomber Jacket     coat         95    0.538201
     P0018    Rose Puff Sleeve Top    shirt         42    0.518634


**<span style="color:green">[Opus 4.8]</span> Design gap — `retrieve()` always returns `top_k`, even when nothing fits:**

Look closely at the scores this cell printed: the *best* match tops out around **0.58**, and further down the top-5 you get a bomber jacket and a puff-sleeve top mixed into a "summer dress" query. `np.argsort(-sims)[:top_k]` **always returns 5 rows** — there is no notion of "good enough." The retriever hands the model its 5 least-bad guesses no matter how weak they are, and the LLM, told to recommend from the list, obligingly recommends them. (You can see this bite in §7: the "cosy, cold-day hiking" query surfaces a *lightweight, breathable linen shirt* as the top pick.)

This is why Sarah's own checklist (§8) says the system prompt must include *"if nothing matches, say so"* — but that rule can't fire when retrieval never admits it found nothing. The standard fix is a **similarity floor**: drop candidates below a threshold and, if none survive, short-circuit to a graceful "I couldn't find a good match" instead of calling the LLM:

```python
def retrieve(query, top_k=5, min_sim=0.3):
    q_emb = retriever.encode([query])
    sims = cosine_similarity(q_emb, catalogue_emb)[0]
    top_idx = [i for i in np.argsort(-sims)[:top_k] if sims[i] >= min_sim]
    out = df.iloc[top_idx].copy()
    out['similarity'] = sims[top_idx]
    return out.reset_index(drop=True)   # may be empty -> caller answers "no good match"
```

Tune `min_sim` on real queries (cosine ≈ 0.3 is a reasonable starting point for MiniLM, but calibrate it — it's model- and domain-specific). This is the retrieval-side complement to the §E1 hallucination check: one stops weak *retrieval*, the other catches weak *generation*.

**<span style="color:green"><em>[Opus 4.8] — end of note</em></span>**

## 5 · Build the augmented prompt

Format the retrieved products into a system prompt that constrains the LLM's behaviour.

In [7]:
def build_rag_prompt(query, retrieved):
    """Compose a system+user message pair.

    Important design choice: we put the catalogue in the USER turn, not the
    system prompt. A 360M-parameter model can drift on long system prompts
    that contain structured data — it tends to treat them as text to continue
    rather than rules to follow. Putting the catalogue in the user turn,
    framed as "here is what we have, answer my question using only these",
    works much more reliably.
    """
    # Format each retrieved product as one readable bullet line for the prompt.
    product_lines = []
    for _, row in retrieved.iterrows():
        product_lines.append(
            f"- [{row['product_id']}] {row['name']} ({row['category']}, £{row['price_gbp']}): {row['description']}"
        )
    catalogue_block = '\n'.join(product_lines)  # join all bullets into one block of text

    # The system prompt sets the rules: ONLY use the listed products — this is what stops hallucination.
    system_prompt = (
        "You are a helpful retail shopping assistant for NorthStar. "
        "You ONLY recommend products that the user has explicitly listed below. "
        "Do not invent products. Be concise — one or two sentences plus the product names."
    )
    # The user turn carries the question PLUS the retrieved catalogue (the "augmented" context).
    user_message = (
        f"Customer question: \"{query}\"\n\n"
        f"Available products from our catalogue:\n{catalogue_block}\n\n"
        "Which one or two would you recommend? Reply with product names and a brief reason."
    )
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_message},
    ]

# Show what the prompt looks like
prompt = build_rag_prompt(question, retrieved)  # the grounded prompt we'll feed the LLM
print('System prompt:')
print(prompt[0]['content'])
print('\nUser message:')
print(prompt[1]['content'])

System prompt:
You are a helpful retail shopping assistant for NorthStar. You ONLY recommend products that the user has explicitly listed below. Do not invent products. Be concise — one or two sentences plus the product names.

User message:
Customer question: "I'm looking for a summer dress under £80. What does NorthStar have?"

Available products from our catalogue:
- [P0007] Sienna Bodycon Dress (dress, £45): Fitted stretch jersey in midnight black. Goes from desk to drinks effortlessly.
- [P0010] Marigold Shift Dress (dress, £52): Sleeveless A-line shift in sunshine yellow. Bold colour for the brave.
- [P0004] Holly Knit Jumper Dress (dress, £72): Cosy ribbed knit dress in oatmeal. Perfect winter layer with thick tights.
- [P0031] Verge Bomber Jacket (coat, £95): Satin baseball-style bomber in midnight. Ribbed cuffs and hem.
- [P0018] Rose Puff Sleeve Top (shirt, £42): Romantic puff-sleeve blouse with smocked waist. Picnic-ready.

Which one or two would you recommend? Reply with pr

## 6 · Send the augmented prompt to the LLM

In [8]:
print(f"Q: {question}\n")
t0 = time.time()
print('A (RAG: retrieve top-5 + ask LLM):\n')
# Same LLM as before, but now its prompt contains the real retrieved products...
answer = llm_chat(prompt, max_new_tokens=200)
print(answer)  # ...so the answer cites genuine NorthStar names/prices instead of inventing them
print(f"\n[generated in {time.time()-t0:.1f}s]")

Q: I'm looking for a summer dress under £80. What does NorthStar have?

A (RAG: retrieve top-5 + ask LLM):

Product 1: [P0007] Sienna Bodycon Dress (dress, £45)
Reason: This dress is perfect for those who love bold colours and effortless movement. It's also stylish enough to be worn as an outfit on its own.

Product 2: [P0010] Marigold Shift Dress (dress, £52)
Reason: The bright color of this dress makes it stand out against any background, making it ideal for casual wear or when you want to add some personality to your outfits.

Product 3: [P0004] Holly Knit Jumper Dress (dress, £72)
Reason: This dress is perfect for those who enjoy layering their clothes together. Its soft fabric and fitted shape make it comfortable to wear over other items like sweaters or cardigans.

Product 4: [P0031] Verge Bomber Jack

[generated in 51.3s]


**Look at the response.** It now references actual products from the retrieved set with their real names and prices — Sienna Bodycon Dress, Marigold Shift Dress, etc. The LLM has stopped hallucinating because we gave it the answer in its context.

This is RAG. The retrieval keeps the model grounded; the LLM does the language understanding and synthesis.

**A practical note on small models:** SmolLM2-360M is tiny by modern standards, and you'll see it occasionally do strange things — copying our catalogue format too literally, mentioning a product from the retrieved list rather than just one. Production deployments use a much bigger model (Claude, GPT-4) and these quirks disappear. The *architecture* is the same; the *quality* is what scale buys you.

## 7 · Wrap it up: a clean RAGSystem class

In [9]:
class RAGSystem:
    """Minimum-viable RAG shopping assistant."""

    def __init__(self, df, retriever, llm, tokenizer):
        # Store the catalogue and both models on the instance so methods can reuse them.
        self.df = df.reset_index(drop=True).copy()
        self.retriever = retriever
        self.llm = llm
        self.tok = tokenizer
        # Pre-compute catalogue embeddings once
        # (done in __init__ so we embed the catalogue a single time, not on every question).
        docs = (self.df['name'] + ' — ' + self.df['description']
                + ' (£' + self.df['price_gbp'].astype(str) + ')').tolist()
        self.embeddings = self.retriever.encode(docs, show_progress_bar=False)

    def retrieve(self, query, top_k=5):
        # Same retrieval logic as the standalone function: embed query, score, take top_k.
        q_emb = self.retriever.encode([query])
        sims = cosine_similarity(q_emb, self.embeddings)[0]   # similarity to every product
        idx = np.argsort(-sims)[:top_k]                       # indices of the top_k highest scores
        return self.df.iloc[idx].copy().assign(similarity=sims[idx]).reset_index(drop=True)

    def _build_prompt(self, query, retrieved):
        # Catalogue goes in the USER turn (see build_rag_prompt above for why).
        lines = [f"- [{r['product_id']}] {r['name']} ({r['category']}, £{r['price_gbp']}): {r['description']}"
                 for _, r in retrieved.iterrows()]
        block = '\n'.join(lines)
        # System rule again constrains the model to the listed products only.
        sys_msg = (
            "You are a helpful retail shopping assistant for NorthStar. "
            "You ONLY recommend products that the user has explicitly listed below. "
            "Do not invent products. Be concise."
        )
        user_msg = (
            f"Customer question: \"{query}\"\n\n"
            f"Available products from our catalogue:\n{block}\n\n"
            "Which one or two would you recommend? Reply with product names and a brief reason."
        )
        return [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': user_msg},
        ]

    def ask(self, query, top_k=5, max_new_tokens=200):
        # The full RAG flow in one call: RETRIEVE -> build prompt -> GENERATE.
        retrieved = self.retrieve(query, top_k=top_k)
        messages = self._build_prompt(query, retrieved)
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = self.tok(prompt, return_tensors='pt')['input_ids']
        out = self.llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=self.tok.eos_token_id,
                                repetition_penalty=1.15)
        answer = self.tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        # Return BOTH the answer and the retrieved products (the audit trail / grounding).
        return {'answer': answer, 'retrieved': retrieved}

# Instantiate and try a few questions
rag = RAGSystem(df, retriever, llm, tokenizer)  # __init__ embeds the catalogue right here
print('RAGSystem ready.')

RAGSystem ready.


### Try several customer questions

In [10]:
# A few different shopping intents to show retrieval adapts to each query.
questions = [
    "I need something cosy to wear hiking on a cold day",
    "What's a smart shirt I can wear to the office?",
    "Recommend a beach holiday outfit for under £150",
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    t0 = time.time()
    result = rag.ask(q)  # runs the full retrieve-then-generate pipeline for this question
    print(f"\nRetrieved products:")
    # Show the top 3 retrieved items — the grounding behind the answer.
    for _, row in result['retrieved'].head(3).iterrows():
        print(f"  - {row['name']:<35s} ({row['category']:<10s} £{row['price_gbp']:>3d})")
    print(f"\nA: {result['answer']}")
    print(f"\n[total: {time.time()-t0:.1f}s]")


Q: I need something cosy to wear hiking on a cold day

Retrieved products:
  - Frost Linen Shirt                   (shirt      £ 55)
  - Aspen Snow Boots                    (footwear   £155)
  - Boulder Hiking Boots                (footwear   £175)

A: Frost Linen Shirt - This is my top choice as it's lightweight, breathable, and perfect for chilly days like this. It also comes with a great price tag of £55.

Aspen Snow Boots - These boots are ideal for those who enjoy winter sports and want some extra protection against the elements. They're made from durable materials and come with a good price point of £155.

Boulder Hiking Boots - If you plan on tackling more challenging trails, these boots will keep your feet warm and dry. The Vibram sole provides excellent traction and they're priced at £175.

Tarn Waxed Jacket - A classic piece of outdoor gear that can withstand even the toughest conditions. The moss material adds an earthy touch while the waterproof nature makes them perfect f

**This is RAG in production form.** ~50 lines of Python wrapping two open-source models. It can answer arbitrary natural-language shopping questions, grounded in actual product data, with audit trails (you can see which products were retrieved for any given answer).

This is the architecture behind:
- ChatGPT's "browsing" / "memory"
- Perplexity AI
- Most enterprise "chat with your docs" products
- Customer-support copilots
- Code-completion tools (the docs are the codebase)

Same pattern, different data.

## 8 · Sarah's production checklist

For a production RAG system at NorthStar, Sarah's notes:

**Sarah's production checklist** — what it takes to run this RAG system for real at NorthStar:

**▸ Retrieval**
- `top_k = 5 to 10` — too few misses good products, too many dilutes the prompt
- Hybrid (TF-IDF + semantic) — same as L09
- Apply category/price filters BEFORE retrieval where the user is specific

**▸ Prompt design**
- System prompt MUST say *"only recommend from the catalogue below"*
- System prompt MUST say *"if nothing matches, say so"* — prevents over-eagerness
- Show product IDs so users can click through

**▸ Safety**
- Post-process: extract product names from the answer, validate against the catalogue
- If the answer mentions a product NOT in the retrieved set: flag for review
- Always show the source products alongside the answer (audit trail)

**▸ Quality**
- Log `(query → retrieved → answer)` tuples for offline evaluation
- Hand-label 100 queries as good/bad, track that metric weekly
- A/B test against the L09 keyword+semantic baseline before launch

**▸ Cost / latency**
- Self-hosted SmolLM2: free but 1-3s/response on CPU, ~200ms on GPU
- Hosted API (Claude Haiku, GPT-4 mini): ~$0.001 per query, 500ms latency, much better quality
- Most production setups use a hosted API for quality; self-host as fallback

## 9 · The Module 3 closing recap

What you've built across nine lessons:

| Lesson | Foundational skill |
|--------|--------------------|
| L01-L03 | Classical ML modelling, evaluation, making it work |
| L04 | Trees, ensembles, hyperparameter tuning |
| L05 | Clustering, dimensionality reduction, anomaly detection |
| L06 | Time-series forecasting with seasonality |
| L07 | Neural networks, the PyTorch training loop |
| L08 | CNNs, transfer learning, computer vision |
| L09 | Sentence embeddings, semantic search, hybrid retrieval |
| L10 | Attention, transformers, LLMs, RAG |

You can ship classical ML systems. You can fine-tune CNNs. You can build semantic search. And as of this notebook, you can build a RAG-powered chat application from scratch.

That's a full-stack production ML engineer's toolkit.

**The lesson Sarah's journey teaches:** every shipping ML system is a chain of decisions about *which problem to solve, which tool to use, and what to measure*. The mathematics is the easy part. The hard part is taste — picking the right tool, being honest about what works, and shipping something that helps real customers.

Welcome to the field. Now go build something.

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · The hallucination check

A simple safety pattern: after generation, check that every product name mentioned in the answer was actually in the retrieved set. If not, flag the response for review.

In [12]:
import re

def hallucination_check(answer, retrieved):
    """Find product names in the answer that don't appear in the retrieved set."""
    # The set of names we actually gave the model for this query.
    retrieved_names = set(retrieved['name'].tolist())
    # Look for product names — naive heuristic: capitalised multi-word phrases
    # In production you'd extract product IDs (P0001 style) which is more robust
    candidates = re.findall(r'\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b', answer)
    candidates = set(candidates)
    catalogue_names = set(df['name'].tolist())  # every real product name in the catalogue

    # Mentioned names that WERE in the retrieved set => correctly grounded.
    in_retrieved = candidates & retrieved_names
    # Mentioned names NOT anywhere in the catalogue => the model invented them (a hallucination).
    unknown = candidates - catalogue_names
    return {
        'mentioned_products': list(candidates),
        'in_retrieved': list(in_retrieved),
        'potentially_invented': list(unknown),
    }

q = "What's a fancy outfit for a wedding?"
result = rag.ask(q)
print(f"Q: {q}\nA: {result['answer']}\n")
check = hallucination_check(result['answer'], result['retrieved'])  # audit the answer for hallucinations
print(f"Mentioned product-like phrases : {check['mentioned_products']}")
print(f"Matched retrieved set          : {check['in_retrieved']}")
print(f"⚠️  Potentially invented        : {check['potentially_invented']}")

Q: What's a fancy outfit for a wedding?
A: I'd recommend the Rose Puff Sleeve Top as it is a romantic and elegant choice perfect for an afternoon event. It also pairs well with your chosen dress to create a beautiful ensemble.

Mentioned product-like phrases : ['Rose Puff Sleeve Top']
Matched retrieved set          : ['Rose Puff Sleeve Top']
⚠️  Potentially invented        : []


## E2 · Re-ranking with the LLM

A more sophisticated retrieval step: retrieve top-20 with embeddings, then ask the LLM to re-rank those 20. The LLM brings deeper understanding but is more expensive.

In [13]:
def llm_rerank(query, retrieved, top_k=3):
    """Ask the LLM to pick the most relevant products from a list.

    Subtle bug to avoid: don't put example product IDs in the prompt
    (e.g. 'Format: P0001, P0042') — small models often copy the example IDs
    instead of picking from the actual options. Use a structural example only.
    """
    # Format the wide candidate set as a list (id: name — short description) for the model to rank.
    options = []
    for _, r in retrieved.iterrows():
        options.append(f"  {r['product_id']}: {r['name']} — {r['description'][:80]}")
    options_str = '\n'.join(options)

    # Ask the LLM to act purely as a relevance ranker and output ONLY product IDs.
    messages = [
        {'role': 'system', 'content': 'You are a product-search relevance ranker.'},
        {'role': 'user',   'content': (
            f"Customer query: {query}\n\n"
            f"Product options:\n{options_str}\n\n"
            f"List the {top_k} most relevant product IDs from above, in order of relevance, comma-separated. "
            "Only output the product IDs, nothing else."
        )},
    ]
    response = llm_chat(messages, max_new_tokens=40)
    # Parse product IDs from the response (any P#### pattern in the model's text).
    all_ids = re.findall(r'P\d{4}', response)
    # Filter to IDs that were actually in the candidate set
    # (guards against the model echoing IDs that weren't among the options).
    candidate_ids = set(retrieved['product_id'].tolist())
    valid_ids = [i for i in all_ids if i in candidate_ids]
    return valid_ids[:top_k], response

q = "Something to wear running on a cold morning"
# Re-ranking flow: retrieve WIDE (top-10) with fast embeddings...
retrieved = rag.retrieve(q, top_k=10)
print(f"Initial top-10 by embeddings:")
print(retrieved[['product_id','name']].to_string(index=False))
print()
# ...then let the (slower, smarter) LLM NARROW that list down to the best 3.
ranked_ids, raw = llm_rerank(q, retrieved, top_k=3)
print(f"\nLLM raw response: {raw!r}")
print(f"LLM re-ranked top-3 (filtered to candidates): {ranked_ids}")
# Name the picks
for pid in ranked_ids:
    name = retrieved.loc[retrieved['product_id']==pid, 'name'].iloc[0]
    print(f"  {pid}: {name}")

Initial top-10 by embeddings:
product_id                   name
     P0061   Drift Running Shorts
     P0041 Cloud Running Trainers
     P0014      Frost Linen Shirt
     P0007   Sienna Bodycon Dress
     P0018   Rose Puff Sleeve Top
     P0033    Cinder Biker Jacket
     P0030   Ember Quilted Jacket
     P0031    Verge Bomber Jacket
     P0028  Glacier Puffer Jacket
     P0043       Loam Ankle Boots


LLM raw response: 'P0061, P0041, P0014'
LLM re-ranked top-3 (filtered to candidates): ['P0061', 'P0041', 'P0014']
  P0061: Drift Running Shorts
  P0041: Cloud Running Trainers
  P0014: Frost Linen Shirt


This pattern — embeddings retrieve a wide candidate set, LLM does precise re-ranking — is what production-grade RAG systems use. The LLM call is more expensive but adds quality where it counts. Most queries get answered well by retrieval alone; the LLM re-rank is a finishing pass.